In [14]:
import cv2
import json
import numpy as np
import glob
import os

curfolder = os.getcwd()
# List folders in a main folder
folderstotrack = glob.glob(os.path.join(curfolder, 'projectdata', '*'))

# Get all folders per participant, per session
pcnfolders = []

for i in folderstotrack:
    pcnfolders_in_session = glob.glob(os.path.join(i, '*'))
    pcnfolders = pcnfolders + pcnfolders_in_session

# There might be some other things we don't want now
pcnfolders = [x for x in pcnfolders if 'Config' not in x]
pcnfolders = [x for x in pcnfolders if 'opensim' not in x]
pcnfolders = [x for x in pcnfolders if 'xml' not in x]
pcnfolders = [x for x in pcnfolders if 'ResultsInverseDynamics' not in x]
pcnfolders = [x for x in pcnfolders if 'ResultsInverseKinematics' not in x]
pcnfolders = [x for x in pcnfolders if 'sto' not in x]
pcnfolders = [x for x in pcnfolders if 'txt' not in x]
pcnfolders = [x for x in pcnfolders if 'calibration' not in x]

print(pcnfolders[0:10])

['f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_11_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_12_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_13_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_14_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_15_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_16_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_17_p1', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_20_p0', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_processing\\projectdata\\Session_10_1\\10_1_21_p0', 'f:\\FLESH_ContinuousBodilyEffort\\02_MotionTracking_p

In [21]:
# ---------------------------------------------------------------------------
# Keypoint pooling + convex hull
# ---------------------------------------------------------------------------

def get_person_points(person, conf_thresh=0.1):
    """
    Returns all confidently-detected (x, y) points for one person as an
    (N, 2) int32 array, or None if too few points are confident.

    For BODY_135 (the single-network whole-body model), ALL 135 points --
    body, feet, both hands, face -- are packed into pose_keypoints_2d as one
    flat array; face_keypoints_2d/hand_*_keypoints_2d are empty. This still
    reads all four keys for robustness in case a differently-configured run
    (e.g. standard BODY_25 + --face --hand) populates the separate arrays
    instead -- but for BODY_135 only pose_keypoints_2d will be non-empty.
    """
    keys = [
        'pose_keypoints_2d',
        'face_keypoints_2d',
        'hand_left_keypoints_2d',
        'hand_right_keypoints_2d',
    ]

    all_points = []
    for key in keys:
        arr = person.get(key)
        if not arr:
            continue
        kp = np.array(arr).reshape(-1, 3)  # x, y, confidence
        all_points.append(kp)

    if not all_points:
        return None

    kp = np.vstack(all_points)
    valid = kp[kp[:, 2] > conf_thresh][:, :2]

    if len(valid) < 3:  # need at least 3 points for a hull
        return None

    return valid.astype(np.int32)


def blur_convex_hull(frame, points, padding_px=20, blur_strength=35):
    """
    Blurs the region inside the convex hull of `points`, expanded outward
    by `padding_px` so the blur fully covers the body silhouette rather
    than clipping right at the joints. Leaves everything outside the hull
    untouched.
    """
    h, w = frame.shape[:2]
    hull = cv2.convexHull(points)

    # Expand hull outward from its centroid by padding_px (roughly — scales
    # with how far each point already is from center, then adds a flat margin)
    centroid = hull.mean(axis=0)
    expanded = []
    for pt in hull[:, 0, :]:
        direction = pt - centroid[0]
        norm = np.linalg.norm(direction)
        if norm > 0:
            direction = direction / norm
        expanded.append(pt + direction * padding_px)
    expanded_hull = np.array(expanded, dtype=np.int32).reshape(-1, 1, 2)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, expanded_hull, 255)

    k = blur_strength if blur_strength % 2 == 1 else blur_strength + 1
    blurred = cv2.GaussianBlur(frame, (k, k), 0)

    mask_3ch = cv2.merge([mask, mask, mask])
    frame[:] = np.where(mask_3ch == 255, blurred, frame)
    return frame


# ---------------------------------------------------------------------------
# Skeleton drawing (BODY_135)
# ---------------------------------------------------------------------------
#
# IMPORTANT: BODY_135 is NOT BODY_25 + separate face/hand arrays. It is a
# single-network whole-body model whose ENTIRE 135-point output lives in
# pose_keypoints_2d as one flat array (face_keypoints_2d / hand_*_keypoints_2d
# are empty). Its joint ordering is also different from BODY_25 (e.g. index 1
# is LEye here, not Neck). Verified directly against OpenPose source
# (src/openpose/pose/poseParameters.cpp, POSE_BODY_135_BODY_PARTS /
# POSE_BODY_PART_PAIRS for BODY_135) and cross-checked against a real sample
# file's y-coordinates (head < shoulders < hips < ankles, as expected).
#
# Body (0-18), feet (19-24): indices match POSE_BODY_135_BODY_PARTS exactly.
# Hands start at H135 = 25 (20 points each: left then right).
# Face starts at F135 = 65 (70 points: contour, eyebrows, nose, eyes, mouth, pupils).

H135 = 25
F135 = 65

BODY135_NAMES = {
    0: "Nose", 1: "LEye", 2: "REye", 3: "LEar", 4: "REar",
    5: "LShoulder", 6: "RShoulder", 7: "LElbow", 8: "RElbow",
    9: "LWrist", 10: "RWrist", 11: "LHip", 12: "RHip",
    13: "LKnee", 14: "RKnee", 15: "LAnkle", 16: "RAnkle",
    17: "UpperNeck", 18: "HeadTop",
    19: "LBigToe", 20: "LSmallToe", 21: "LHeel",
    22: "RBigToe", 23: "RSmallToe", 24: "RHeel",
}

# Body + foot bones: ONLY real anatomical connections. The OpenPose source
# pair list mixes true bones with "redundant" connections it uses internally
# for assembly-graph robustness (e.g. LWrist-RWrist, LEar-REar, wrist-to-hip)
# -- those aren't bones and are deliberately excluded here.
BODY135_BODY_PAIRS = [
    (0, 1), (0, 2),             # Nose-LEye, Nose-REye
    (1, 3), (2, 4),             # LEye-LEar, REye-REar
    (17, 18),                   # UpperNeck-HeadTop
    (5, 17), (6, 17),           # LShoulder/RShoulder-UpperNeck (both sides, real connection point)
    (5, 7), (7, 9),             # LShoulder-LElbow-LWrist
    (6, 8), (8, 10),            # RShoulder-RElbow-RWrist
    (5, 11), (6, 12),           # LShoulder-LHip, RShoulder-RHip
    (11, 13), (13, 15),         # LHip-LKnee-LAnkle
    (12, 14), (14, 16),         # RHip-RKnee-RAnkle
    (15, 19), (19, 20), (15, 21),  # LAnkle-LBigToe-LSmallToe, LAnkle-LHeel
    (16, 22), (22, 23), (16, 24),  # RAnkle-RBigToe-RSmallToe, RAnkle-RHeel
]

# Hand bones, re-based onto absolute indices using H135 offset (left hand: H135+0..19, right: H135+20..39)
BODY135_LEFT_HAND_PAIRS = [
    (9, H135+0), (H135+0, H135+1), (H135+1, H135+2), (H135+2, H135+3),
    (9, H135+4), (H135+4, H135+5), (H135+5, H135+6), (H135+6, H135+7),
    (9, H135+8), (H135+8, H135+9), (H135+9, H135+10), (H135+10, H135+11),
    (9, H135+12), (H135+12, H135+13), (H135+13, H135+14), (H135+14, H135+15),
    (9, H135+16), (H135+16, H135+17), (H135+17, H135+18), (H135+18, H135+19),
]
BODY135_RIGHT_HAND_PAIRS = [
    (10, H135+20), (H135+20, H135+21), (H135+21, H135+22), (H135+22, H135+23),
    (10, H135+24), (H135+24, H135+25), (H135+25, H135+26), (H135+26, H135+27),
    (10, H135+28), (H135+28, H135+29), (H135+29, H135+30), (H135+30, H135+31),
    (10, H135+32), (H135+32, H135+33), (H135+33, H135+34), (H135+34, H135+35),
    (10, H135+36), (H135+36, H135+37), (H135+37, H135+38), (H135+38, H135+39),
]

# Face bones, re-based onto absolute indices using F135 offset
BODY135_FACE_PAIRS = [
    # Nose-tip / eye anchors back to the COCO-style face points
    (0, F135+30), (2, F135+39), (1, F135+42),
    # Contour (jawline)
    (F135+0, F135+1), (F135+1, F135+2), (F135+2, F135+3), (F135+3, F135+4),
    (F135+4, F135+5), (F135+5, F135+6), (F135+6, F135+7), (F135+7, F135+8),
    (F135+8, F135+9), (F135+9, F135+10), (F135+10, F135+11), (F135+11, F135+12),
    (F135+12, F135+13), (F135+13, F135+14), (F135+14, F135+15), (F135+15, F135+16),
    # Contour-eyebrow + eyebrows
    (F135+0, F135+17), (F135+16, F135+26), (F135+17, F135+18), (F135+18, F135+19),
    (F135+19, F135+20), (F135+20, F135+21), (F135+21, F135+22), (F135+22, F135+23),
    (F135+23, F135+24), (F135+24, F135+25), (F135+25, F135+26),
    # Eyebrow-nose + nose
    (F135+21, F135+27), (F135+22, F135+27), (F135+27, F135+28), (F135+28, F135+29),
    (F135+29, F135+30), (F135+30, F135+33), (F135+33, F135+32), (F135+32, F135+31),
    (F135+33, F135+34), (F135+34, F135+35),
    # Nose-eyes + eyes
    (F135+27, F135+39), (F135+27, F135+42), (F135+36, F135+37), (F135+37, F135+38),
    (F135+38, F135+39), (F135+39, F135+40), (F135+40, F135+41),
    (F135+42, F135+43), (F135+43, F135+44), (F135+44, F135+45), (F135+45, F135+46), (F135+46, F135+47),
    # Nose-mouth + outer mouth
    (F135+33, F135+51), (F135+48, F135+49), (F135+49, F135+50), (F135+50, F135+51),
    (F135+51, F135+52), (F135+52, F135+53), (F135+53, F135+54), (F135+54, F135+55),
    (F135+55, F135+56), (F135+56, F135+57), (F135+57, F135+58), (F135+58, F135+59),
    # Outer-inner + inner mouth
    (F135+48, F135+60), (F135+54, F135+64), (F135+60, F135+61), (F135+61, F135+62),
    (F135+62, F135+63), (F135+63, F135+64), (F135+64, F135+65), (F135+65, F135+66), (F135+66, F135+67),
    # Eyes-pupils
    (F135+36, F135+68), (F135+39, F135+68), (F135+42, F135+69), (F135+45, F135+69),
]

SKELETON_COLOR = (0, 255, 255)   # cyan-ish, high contrast on blurred bg
JOINT_COLOR = (0, 0, 255)        # red dots
HAND_COLOR = (255, 200, 0)
FACE_COLOR = (0, 255, 0)


def draw_skeleton(frame, person, conf_thresh=0.1):
    """
    Draws the BODY_135 skeleton (body+feet, both hands, face) directly from
    pose_keypoints_2d -- the single flat 135-point array. Independent of any
    pre-rendered OpenPose video, so it stays crisp regardless of blurring
    applied underneath.
    """
    arr = person.get('pose_keypoints_2d')
    if not arr:
        return frame
    kp = np.array(arr).reshape(-1, 3)
    n = len(kp)

    def draw_pairs(pairs, color, thickness):
        for a, b in pairs:
            if a < n and b < n and kp[a, 2] > conf_thresh and kp[b, 2] > conf_thresh:
                pa = (int(kp[a, 0]), int(kp[a, 1]))
                pb = (int(kp[b, 0]), int(kp[b, 1]))
                cv2.line(frame, pa, pb, color, thickness, cv2.LINE_AA)

    draw_pairs(BODY135_BODY_PAIRS, SKELETON_COLOR, 2)
    draw_pairs(BODY135_LEFT_HAND_PAIRS, HAND_COLOR, 1)
    draw_pairs(BODY135_RIGHT_HAND_PAIRS, HAND_COLOR, 1)
    draw_pairs(BODY135_FACE_PAIRS, FACE_COLOR, 1)

    # Joint dots for the main body (skip hand/face dots -- too dense to read)
    for idx in range(19):
        x, y, c = kp[idx]
        if c > conf_thresh:
            cv2.circle(frame, (int(x), int(y)), 3, JOINT_COLOR, -1, cv2.LINE_AA)

    return frame


# ---------------------------------------------------------------------------
# Per-camera processing: raw video + JSON -> blurred body + skeleton overlay
# ---------------------------------------------------------------------------

def process_video_with_keypoints(raw_video_path, json_folder, output_path,
                                   padding_px=20, blur_strength=35, fourcc_str='XVID'):
    """
    Reads the RAW (unannotated) video, and for each frame:
      1) blurs the convex-hull body region for each detected person
      2) draws that person's skeleton on top from the JSON keypoints
    Returns output_path on success, None if the video couldn't be opened.
    """
    cap = cv2.VideoCapture(raw_video_path)
    if not cap.isOpened():
        print(f'  Could not open video: {raw_video_path}')
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    json_files = sorted(glob.glob(os.path.join(json_folder, '*_keypoints.json')))
    n_json = len(json_files)

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx < n_json:
            with open(json_files[frame_idx], 'r') as f:
                data = json.load(f)

            people = data.get('people', [])

            # Pass 1: blur every detected person's body region first
            for person in people:
                points = get_person_points(person)
                if points is None:
                    continue
                frame = blur_convex_hull(frame, points, padding_px=padding_px,
                                          blur_strength=blur_strength)

            # Pass 2: draw skeletons on top, after all blurring is done,
            # so one person's skeleton never gets blurred by another's mask
            for person in people:
                frame = draw_skeleton(frame, person)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

    if frame_idx > n_json:
        print(f'  Note: video had {frame_idx} frames but only {n_json} JSON files '
              f'— frames {n_json}-{frame_idx-1} were written without blur/skeleton.')

    print(f'  Processed {raw_video_path} -> {output_path} ({frame_idx} frames)')
    return output_path


# ---------------------------------------------------------------------------
# Concatenation
# ---------------------------------------------------------------------------

def concat_videos_horizontally(video_paths, output_path, target_height=None, fourcc_str='XVID'):
    """
    Stack 2+ videos side by side into one video. Videos may have different
    native resolutions/frame counts; each is resized to a common height
    (preserving aspect ratio) and the combined video runs only as long as
    the shortest input.
    """
    caps = [cv2.VideoCapture(p) for p in video_paths]
    for p, c in zip(video_paths, caps):
        if not c.isOpened():
            raise RuntimeError(f'Could not open {p} for concatenation')

    fps_list = [c.get(cv2.CAP_PROP_FPS) or 25.0 for c in caps]
    fps = min(fps_list)
    if len(set(round(f, 2) for f in fps_list)) > 1:
        print(f'  Note: input fps differ {fps_list} — using {fps} for output.')

    heights = [int(c.get(cv2.CAP_PROP_FRAME_HEIGHT)) for c in caps]
    widths = [int(c.get(cv2.CAP_PROP_FRAME_WIDTH)) for c in caps]

    if target_height is None:
        target_height = min(heights)

    target_widths = [int(w * (target_height / h)) for w, h in zip(widths, heights)]
    combined_width = sum(target_widths)

    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
    out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, target_height))

    frame_count = 0
    while True:
        frames = []
        all_ok = True
        for c, tw in zip(caps, target_widths):
            ret, frame = c.read()
            if not ret:
                all_ok = False
                break
            frame = cv2.resize(frame, (tw, target_height))
            frames.append(frame)

        if not all_ok:
            break

        combined = np.hstack(frames)
        out.write(combined)
        frame_count += 1

    for c in caps:
        c.release()
    out.release()

    print(f'  Concatenated {len(video_paths)} videos -> {output_path} ({frame_count} frames)')


# ---------------------------------------------------------------------------
# Main driver
# ---------------------------------------------------------------------------

def process_session(session_folder, raw_video_subfolder='raw-2d', padding_px=20,
                     blur_strength=35, cleanup_temp=False):
    """
    session_folder: path to one participant/session folder, expected to contain
        <raw_video_subfolder>/*.avi    (3 RAW camera videos, pre-OpenPose)
        pose/pose_cam1_json, pose_cam2_json, pose_cam3_json
    Writes:
        pose-2d-masked/maskedN.avi     (per-camera blurred-body + skeleton videos)
        pose-2d-masked/combined.avi    (3 views side by side)
    """
    print(f'Processing {session_folder}')

    raw_videos = sorted(glob.glob(os.path.join(session_folder, raw_video_subfolder, '*.avi')))
    if len(raw_videos) != 3:
        print(f'  Expected 3 raw videos in {raw_video_subfolder}/, found {len(raw_videos)} — skipping.')
        return

    json_folders = [
        os.path.join(session_folder, 'pose', 'pose_cam1_json'),
        os.path.join(session_folder, 'pose', 'pose_cam2_json'),
        os.path.join(session_folder, 'pose', 'pose_cam3_json'),
    ]

    masked_folder = os.path.join(session_folder, 'pose-2d-masked')
    os.makedirs(masked_folder, exist_ok=True)

    combined_path = os.path.join(masked_folder, 'combined.avi')
    if os.path.exists(combined_path):
        print('  combined.avi already exists — skipping session.')
        return

    masked_paths = []
    for it, (video_path, json_folder) in enumerate(zip(raw_videos, json_folders)):
        out_path = os.path.join(masked_folder, f'masked{it}.avi')
        result = process_video_with_keypoints(
            video_path, json_folder, out_path,
            padding_px=padding_px, blur_strength=blur_strength,
        )
        if result:
            masked_paths.append(result)

    if len(masked_paths) != 3:
        print('  Not all three cameras processed successfully — skipping concatenation.')
        return

    concat_videos_horizontally(masked_paths, combined_path)

    if cleanup_temp:
        for p in masked_paths:
            os.remove(p)

In [ ]:
raw_video_subfolder = 'raw-2d'  # subfolder holding the ORIGINAL videos, per your first script
padding_px = 20             # how far the blur extends past the convex hull, in pixels
blur_strength = 35          # Gaussian kernel size (odd number) -- higher = blurrier
cleanup_temp = False        # set True to delete the per-camera videos, keep only combined.avi

for session in pcnfolders:
    print(session)
    process_session(session, raw_video_subfolder=raw_video_subfolder,
                        padding_px=padding_px, blur_strength=blur_strength,
                        cleanup_temp=cleanup_temp)
    break

f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1
Processing f:\FLESH_ContinuousBodilyEffort\02_MotionTracking_processing\projectdata\Session_10_1\10_1_11_p1
